In [1]:
# ============================================================
# Notebook 3 — Cell 1: Install
# Run once. Then Runtime → Restart session. Then Cell 2.
# ============================================================
!pip install -q streamlit pyngrok pydub openai-whisper \
    sentence-transformers faiss-cpu rank-bm25 scikit-learn \
    groq google-genai
!pip install -q "pyannote.audio==3.1.1"
!pip install -q numpy==1.26.4

print("✅ Done — now Runtime → Restart session, then run Cell 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ============================================================
# Fix torchaudio compatibility for pyannote
# ============================================================
!pip install -q "torchaudio==2.0.2" --index-url https://download.pytorch.org/whl/cpu
!pip install -q "pyannote.audio==3.1.1"
!pip install -q numpy==1.26.4

print("✅ Fixed — now Runtime → Restart session, then run Cell 2, then Cell 3")

ERROR: Could not find a version that satisfies the requirement torchaudio==2.0.2 (from versions: 2.2.0+cpu, 2.2.1+cpu, 2.2.2+cpu, 2.3.0+cpu, 2.3.1+cpu, 2.4.0+cpu, 2.4.1+cpu, 2.5.0+cpu, 2.5.1+cpu, 2.6.0+cpu, 2.7.0+cpu, 2.7.1+cpu, 2.8.0+cpu, 2.9.0+cpu, 2.9.1+cpu, 2.10.0+cpu, 2.11.0+cpu)
ERROR: No matching distribution found for torchaudio==2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
ERROR: pip's depen

In [1]:
# ============================================================
# Notebook 3 — Cell 2: Setup
# ============================================================
from google.colab import drive, userdata
import os, json, sys

drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/MeetingMind_v2"

CONFIG = {
    "root"            : ROOT,
    "transcripts_dir" : f"{ROOT}/outputs/transcripts",
    "indices_dir"     : f"{ROOT}/outputs/indices",
    "new_meetings_dir": f"{ROOT}/outputs/new_meetings",
    "embedding_model" : "sentence-transformers/multi-qa-mpnet-base-dot-v1",
    "whisper_model"   : "small",  # CPU-friendly. Use "medium" on GPU.
}

for d in [CONFIG["transcripts_dir"], CONFIG["indices_dir"],
          CONFIG["new_meetings_dir"]]:
    os.makedirs(d, exist_ok=True)

# API keys (read from Colab Secrets)
HF_TOKEN       = userdata.get("HF_TOKEN")
GROQ_API_KEY   = userdata.get("GROQ_API_KEY")
GROQ_API_KEY_2 = userdata.get("GROQ_API_KEY_2") or ""
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""

print("✅ Setup done.")
print(f"   HF token: {'✅' if HF_TOKEN else '❌'}")
print(f"   Groq:     {'✅' if GROQ_API_KEY else '❌'}")
print(f"   Existing meetings on Drive:")
for f in os.listdir(CONFIG["transcripts_dir"]):
    if f.endswith("_master.json"):
        print(f"     • {f.replace('_master.json','')}")

Mounted at /content/drive
✅ Setup done.
   HF token: ✅
   Groq:     ✅
   Existing meetings on Drive:
     • IS1009a
     • EN2001b
     • EN2001a


In [2]:
# ============================================================
# Notebook 3 — Cell 4: Write app.py
# Lightweight Streamlit — NO pyannote, NO whisper.
# Only loads small embedding model + uses Groq API.
# ============================================================

APP = r'''
import streamlit as st
import os, json, time, pickle, re
import numpy as np
import faiss
from rank_bm25 import BM25Okapi

st.set_page_config(page_title="MeetingMind v2", page_icon="🎙️", layout="wide")

ROOT = "/content/drive/MyDrive/MeetingMind_v2"
CONFIG = {
    "transcripts_dir": f"{ROOT}/outputs/transcripts",
    "indices_dir"    : f"{ROOT}/outputs/indices",
    "embedding_model": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
}

MEETING_LABELS = {
    "EN2001a": "EN2001a — AMI Product Design (A)",
    "EN2001b": "EN2001b — AMI Product Design (B)",
    "IS1009a": "IS1009a — AMI Industrial Scenario",
}

@st.cache_resource(show_spinner="Loading embedding model (~30s)...")
def load_embed():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(CONFIG["embedding_model"])

def call_llm(prompt, system="You are a helpful assistant.",
             max_tokens=1024, temperature=0.3):
    from groq import Groq
    groq_key  = os.environ.get("GROQ_API_KEY", "")
    groq_key2 = os.environ.get("GROQ_API_KEY_2", "") or groq_key
    gemini_key = os.environ.get("GEMINI_API_KEY", "") or None

    attempts = []
    if groq_key:  attempts.append(("groq1", groq_key, "llama-3.3-70b-versatile"))
    if gemini_key:
        attempts.append(("gemini", gemini_key, "gemini-2.0-flash"))
    if groq_key2 and groq_key2 != groq_key:
        attempts.append(("groq2", groq_key2, "llama-3.3-70b-versatile"))
    if groq_key:  attempts.append(("groq_s", groq_key, "llama3-8b-8192"))

    for name, key, model in attempts:
        try:
            if name == "gemini":
                from google import genai
                client = genai.Client(api_key=key)
                r = client.models.generate_content(
                    model=model, contents=f"{system}\n\n{prompt}")
                return r.text, model
            else:
                client = Groq(api_key=key)
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role":"system","content":system},
                              {"role":"user","content":prompt}],
                    max_tokens=max_tokens, temperature=temperature)
                return r.choices[0].message.content, model
        except Exception:
            time.sleep(2)
            continue
    return "LLM unavailable — check API keys.", "none"

@st.cache_resource
def load_meeting(meeting_id):
    t_dir = CONFIG["transcripts_dir"]
    i_dir = CONFIG["indices_dir"]
    mp = f"{t_dir}/{meeting_id}_master.json"
    if not os.path.exists(mp):
        return None, None
    with open(mp) as f:
        master = json.load(f)
    paths = {
        "sf": f"{i_dir}/{meeting_id}_smart.faiss",
        "nf": f"{i_dir}/{meeting_id}_naive.faiss",
        "sm": f"{i_dir}/{meeting_id}_smart_meta.json",
        "nm": f"{i_dir}/{meeting_id}_naive_meta.json",
        "sb": f"{i_dir}/{meeting_id}_smart_bm25.pkl",
        "nb": f"{i_dir}/{meeting_id}_naive_bm25.pkl",
    }
    if not all(os.path.exists(p) for p in paths.values()):
        st.error(f"Indices missing for {meeting_id} — run Cell 3 to rebuild.")
        return None, master
    indices = {
        "smart_faiss": faiss.read_index(paths["sf"]),
        "naive_faiss": faiss.read_index(paths["nf"]),
    }
    with open(paths["sm"]) as f: indices["smart_meta"] = json.load(f)
    with open(paths["nm"]) as f: indices["naive_meta"] = json.load(f)
    with open(paths["sb"], "rb") as f: indices["smart_bm25"] = pickle.load(f)
    with open(paths["nb"], "rb") as f: indices["naive_bm25"] = pickle.load(f)
    return indices, master

CONV_HIST = []
def clear_history():
    global CONV_HIST
    CONV_HIST = []

POSITION_MAP = {"first":"SPEAKER_00","second":"SPEAKER_01","third":"SPEAKER_02",
                "fourth":"SPEAKER_03"}

def detect_speaker(query):
    q = query.lower()
    m = re.search(r"speaker[_\s]?(\d+)", q)
    if m:
        return f"SPEAKER_{int(m.group(1)):02d}"
    for w, s in POSITION_MAP.items():
        if w in q:
            return s
    return None

def hybrid_retrieve(query, indices, k=5, speaker=None, alpha=0.5):
    embed = load_embed()
    q_emb = embed.encode([query], normalize_embeddings=True).astype("float32")
    sm = indices["smart_meta"]; sf = indices["smart_faiss"]; sb = indices["smart_bm25"]
    if speaker:
        positions = [i for i, m in enumerate(sm) if speaker in m.get("speakers", [])]
        if not positions:
            positions = list(range(len(sm)))
    else:
        positions = list(range(len(sm)))
    if not positions:
        return []
    vecs = np.vstack([sf.reconstruct(i) for i in positions]).astype("float32")
    vsc = (vecs @ q_emb.T).flatten()
    bsc = np.array([sb.get_scores(query.lower().split())[i] for i in positions])
    def _n(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)
    combined = alpha * _n(vsc) + (1 - alpha) * _n(bsc)
    top = np.argsort(combined)[::-1][:k]
    return [sm[positions[i]] for i in top]

def naive_retrieve(query, indices, k=5):
    embed = load_embed()
    q_emb = embed.encode([query], normalize_embeddings=True).astype("float32")
    _, I = indices["naive_faiss"].search(q_emb, k)
    return [indices["naive_meta"][i] for i in I[0] if i >= 0]

def expand_query(query):
    prompt = ("Write 3 alternative phrasings of this question, "
              "one per line, no numbering:\n" + query)
    resp, _ = call_llm(prompt, max_tokens=150, temperature=0.5)
    alts = [l.strip() for l in resp.split("\n") if l.strip() and "?" in l][:3]
    return [query] + alts

def rerank(query, chunks, top_k=5):
    if len(chunks) <= top_k:
        return chunks
    txt = "\n".join(f"[{i}] {c['text'][:200]}" for i, c in enumerate(chunks))
    p = (f"Query: {query}\n\nChunks:\n{txt}\n\n"
         f"Score 1-10. Reply ONLY like: 0:8 1:3 2:9")
    resp, _ = call_llm(p, max_tokens=100, temperature=0.1)
    sc = {int(m.group(1)): int(m.group(2)) for m in re.finditer(r"(\d+):(\d+)", resp)}
    return [chunks[i] for i in sorted(range(len(chunks)),
                                       key=lambda i: sc.get(i, 5), reverse=True)[:top_k]]

def answer_question(query, indices, mode="full", k=5, profiles=None):
    speaker = detect_speaker(query)
    t0 = time.time()
    if mode == "naive":
        chunks = naive_retrieve(query, indices, k)
    elif mode == "smart":
        chunks = hybrid_retrieve(query, indices, k, speaker)
    else:
        seen, all_c = set(), []
        for q in expand_query(query):
            for c in hybrid_retrieve(q, indices, k, speaker):
                key = c.get("chunk_index", c["text"][:30])
                if key not in seen:
                    seen.add(key); all_c.append(c)
        chunks = rerank(query, all_c, k)
    ctx = "\n\n".join(
        f"[{c.get('primary_speaker','?')} | t={c.get('start_time',0):.0f}s]\n{c['text']}"
        for c in chunks)
    hist = ""
    if CONV_HIST:
        hist = "Conversation so far:\n" + "\n".join(
            f"Q: {h['q']}\nA: {h['a'][:200]}" for h in CONV_HIST[-3:]) + "\n\n"
    prof = ""
    if profiles and speaker and speaker in profiles:
        prof = f"Speaker profile: {profiles[speaker]}\n\n"
    prompt = (f"{hist}{prof}Meeting context:\n{ctx}\n\n"
              f"Question: {query}\n\n"
              f"Answer using only the context above. Cite speakers when relevant. Be specific.")
    answer, model_used = call_llm(prompt, max_tokens=512)
    CONV_HIST.append({"q": query, "a": answer})
    if len(CONV_HIST) > 6:
        CONV_HIST.pop(0)
    return {"answer": answer, "chunks": chunks, "model_used": model_used,
            "elapsed_s": round(time.time() - t0, 1), "speaker": speaker}

def generate_report(master):
    mid = master["meeting_id"]; turns = master["turns"]
    spk_stats = master.get("speaker_stats", {})
    total_w = sum(s["words"] for s in spk_stats.values()) or 1
    lines = "\n".join(f"- {s}: {v['turns']} turns, {v['words']} words "
                      f"({100*v['words']//total_w}%)"
                      for s, v in sorted(spk_stats.items()))
    sample = " ".join(t["text"] for t in turns if t["speaker"] != "UNKNOWN")[:3000]
    p = (f"Write a structured meeting report for {mid}.\n\n"
         f"Speakers:\n{lines}\n\nTranscript excerpt:\n{sample}\n\n"
         f"Include: Executive Summary, Key Topics, Decisions, "
         f"Action Items, Speaker Contributions. Use bullets. Be specific.")
    r, _ = call_llm(p, max_tokens=1024)
    return r

def build_profiles(master):
    profs = {}
    for spk in master.get("speakers", []):
        if spk == "UNKNOWN":
            continue
        t = [x for x in master["turns"] if x["speaker"] == spk]
        if not t:
            continue
        sample = " ".join(x["text"] for x in t[:20])[:1500]
        p = (f"Based on these contributions from {spk}, write a 3-sentence "
             f"profile (role, communication style, main topics):\n\n{sample}")
        pf, _ = call_llm(p, max_tokens=150)
        profs[spk] = pf
    return profs

# ============================================================
# UI
# ============================================================

def discover_meetings():
    """Find every _master.json on Drive."""
    out = {}
    t_dir = CONFIG["transcripts_dir"]
    if not os.path.isdir(t_dir):
        return out
    for f in sorted(os.listdir(t_dir)):
        if f.endswith("_master.json"):
            mid = f.replace("_master.json", "")
            out[mid] = MEETING_LABELS.get(mid, mid)
    return out

with st.sidebar:
    st.title("🎙️ MeetingMind v2")
    st.caption("Speaker-Aware Meeting Intelligence")
    st.divider()
    st.subheader("📁 Meetings")
    meetings = discover_meetings()
    if not meetings:
        st.warning("No meetings found. Run Cell 3 to process one.")
    for mid, label in meetings.items():
        if st.button(f"📂 {label}", key=f"l_{mid}", use_container_width=True):
            st.session_state.active = mid
            st.session_state.profiles = {}
            st.session_state.report = None
            st.session_state.qa = []
            clear_history()
            st.rerun()
    if st.button("🔄 Refresh list"):
        st.rerun()
    st.divider()
    st.subheader("⚙️ Settings")
    mode = st.selectbox("Mode", ["full", "smart", "naive"])
    k = st.slider("Chunks", 3, 10, 5)

for key, default in [("active", None), ("profiles", {}),
                     ("report", None), ("qa", [])]:
    if key not in st.session_state:
        st.session_state[key] = default

st.title("🎙️ MeetingMind v2")

if not st.session_state.active:
    st.info("👈 Pick a meeting from the sidebar to start.")
    st.markdown(
        "**To add a new meeting:**\n"
        "1. Go to the Colab notebook\n"
        "2. Open Cell 3 — set `AUDIO_PATH` and `MEETING_ID`\n"
        "3. Run Cell 3 (it shows progress in the notebook)\n"
        "4. Come back here and click 🔄 Refresh list"
    )
else:
    mid = st.session_state.active
    indices, master = load_meeting(mid)
    if not master:
        st.error("Failed to load meeting.")
        st.stop()

    st.subheader(f"📋 {MEETING_LABELS.get(mid, mid)}")
    spk_stats = master.get("speaker_stats", {})
    total_w   = sum(s["words"] for s in spk_stats.values()) or 1

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Turns",        master["total_turns"])
    c2.metric("Words",        f"{master['total_words']:,}")
    c3.metric("Speakers",     len(master.get("speakers", [])))
    c4.metric("Smart chunks", len(indices["smart_meta"]) if indices else 0)

    t_qa, t_tr, t_sp, t_re = st.tabs(
        ["💬 Q&A", "📄 Transcript", "👥 Speakers", "📊 Report"])

    with t_qa:
        for qa in st.session_state.qa:
            with st.chat_message("user"):
                st.write(qa["q"])
            with st.chat_message("assistant"):
                st.write(qa["a"])
                cc1, cc2, cc3 = st.columns(3)
                cc1.caption(f"Mode: {qa['mode']}")
                cc2.caption(f"Model: {qa['model']}")
                cc3.caption(f"Time: {qa['t']}s")
                if qa.get("speaker"):
                    st.caption(f"🔍 Speaker: {qa['speaker']}")
                with st.expander("Retrieved chunks"):
                    for c in qa["chunks"]:
                        st.markdown(
                            f"**{c.get('primary_speaker','?')}** @ "
                            f"{c.get('start_time',0):.0f}s\n\n{c['text'][:300]}")
        q = st.chat_input("Ask a question...")
        if q:
            with st.spinner("Thinking..."):
                r = answer_question(q, indices, mode=mode, k=k,
                                    profiles=st.session_state.profiles)
            st.session_state.qa.append({
                "q": q, "a": r["answer"], "mode": mode,
                "model": r["model_used"], "t": r["elapsed_s"],
                "speaker": r["speaker"], "chunks": r["chunks"]})
            st.rerun()
        if st.button("🗑️ Clear"):
            st.session_state.qa = []
            clear_history()
            st.rerun()

    with t_tr:
        turns = master["turns"]
        spks = sorted(set(t["speaker"] for t in turns if t["speaker"] != "UNKNOWN"))
        flt = st.multiselect("Filter:", spks, default=spks)
        f_turns = [t for t in turns if t["speaker"] in flt]
        ps = 50
        n_p = max(1, (len(f_turns) - 1) // ps + 1)
        st.caption(f"{len(f_turns)} turns — {n_p} pages")
        p = st.number_input(f"Page (1–{n_p})", 1, n_p, 1) - 1
        for t in f_turns[p*ps:(p+1)*ps]:
            ts = f"{int(t['start']//60)}:{int(t['start']%60):02d}"
            st.markdown(f"**{t['speaker']}** `{ts}` *({t['word_count']} words)*\n\n{t['text']}")
            st.divider()

    with t_sp:
        for spk in sorted(spk_stats.keys()):
            if spk == "UNKNOWN":
                continue
            s = spk_stats[spk]
            pct = 100 * s["words"] / total_w
            st.markdown(f"**{spk}** — {s['turns']} turns, {s['words']:,} words ({pct:.1f}%)")
            st.progress(pct / 100)
        st.divider()
        if not st.session_state.profiles:
            if st.button("🤖 Generate Profiles"):
                with st.spinner("Generating..."):
                    st.session_state.profiles = build_profiles(master)
                st.rerun()
        else:
            for spk, pf in st.session_state.profiles.items():
                with st.expander(f"👤 {spk}"):
                    st.write(pf)

    with t_re:
        if not st.session_state.report:
            if st.button("📝 Generate Report", type="primary"):
                with st.spinner("Generating (~30s)..."):
                    st.session_state.report = generate_report(master)
                st.rerun()
        else:
            st.text_area("Report", st.session_state.report, height=600)
            cc1, cc2 = st.columns([1, 5])
            with cc1:
                st.download_button("⬇️ Download", st.session_state.report,
                                   file_name=f"{mid}_report.txt", mime="text/plain")
            with cc2:
                if st.button("🔄 Regenerate"):
                    st.session_state.report = None
                    st.rerun()
'''

with open("/content/app.py", "w") as f:
    f.write(APP)

print(f"✅ app.py written ({len(APP):,} chars)")

✅ app.py written (15,173 chars)


In [ ]:
# ============================================================
# Notebook 3 — Cell 5: Launch (FIXED)
# ============================================================
from pyngrok import ngrok
import subprocess, time, os

NGROK_TOKEN = ""

ngrok.set_auth_token(NGROK_TOKEN)

# ✅ Step 1: Kill everything cleanly
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
ngrok.kill()
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(5)  # wait longer

# ✅ Step 2: Start the app
env = os.environ.copy()
env["GROQ_API_KEY"]   = GROQ_API_KEY   or ""
env["GROQ_API_KEY_2"] = GROQ_API_KEY_2
env["GEMINI_API_KEY"] = GEMINI_API_KEY

proc = subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env,
)
time.sleep(5)

# ✅ Step 3: Open tunnel
tunnel = ngrok.connect(8501)
print(f"✅ Live at: {tunnel.public_url}")

✅ Live at: https://designing-favorite-copper.ngrok-free.dev
